# Tools

A separate experiment, independent of training and evaluation: the assistant calls a tool
through the model's own chat template. Qwen3.5 puts the tool schemas into the system turn
and emits a call as a `<tool_call>` block; the code parses the block, runs the tool and
gets the result back.

`src/tools.py` is the registry. A tool is a JSON schema in `schemas` plus an entry in `executors`
that runs it. The only tool so far is `draw`, backed by Z-Image-Turbo, a 6B image model.
Adding another tool means adding one schema and one function; nothing else changes.

The system prompt is the neutral one from the main experiment plus one sentence: if the
student asks to draw, call `draw` and describe the picture the way a painter would see it.

In [ ]:
import sys
sys.path.insert(0, "..")

from IPython.display import Image, display

from src import data, tools
from src import model as m

model, tokenizer = m.load()

## What the model sees

The schema is rendered by the template, not by us.

In [ ]:
print(tokenizer.apply_chat_template(data.prompt("Нарисуй схему спроса и предложения, никак не могу представить", tools.system),
                                    tools=tools.schemas, add_generation_prompt=True, enable_thinking=False, tokenize=False))

## Ask

One request. Change the text and rerun: nothing was trained for this, the base model decides
on its own whether to call the tool and what to put in the prompt.

In [ ]:
request = "Нарисуй, как выглядит равновесие спроса и предложения, чтобы запомнить картинкой"

raw = m.generate(model, tokenizer, [data.prompt(request, tools.system)], tools=tools.schemas, max_new_tokens=200)[0]
print("ANSWER:", tools.plain(raw))
print("CALLS: ", tools.calls(raw))

## Run the tool

Z-Image-Turbo is loaded once with sequential CPU offload, so it fits next to the 9B model
on one card. Nine steps, no classifier-free guidance: that is how the turbo model was distilled.

In [ ]:
run = tools.executors(tools.Painter())
for name, args in tools.calls(raw):
    path = run[name](args)
    print(name, args)
    display(Image(str(path), width=480))

## Adding a tool

1. Describe it in `src/tools.py` as another schema in `schemas`: name, description, parameters.
2. Add an entry to `executors`: a function of the parsed arguments that returns something printable.
3. Rerun the cells above; the template renders every schema in `schemas`, and `tools.calls` parses any name.

For a multi-turn conversation with the tool results fed back to the model see `chat.ipynb`.